# Inhibitor Operational Benchmark

This notebook runs a configurable **progressive burst-concurrency benchmark**
against the live Inhibitor `/check` API and produces deterministic,
run-specific evidence artifacts.

The benchmark measures:

- successful-response latency distributions;
- successful and attempted throughput;
- transport and API-response reliability;
- configured response-shape anomalies;
- behavior across configured concurrency stages;
- repeatability when multiple trials are configured; and
- short-window post-burst recovery behavior.

It does not measure sustained-load endurance, soak behavior, production
autoscaling, disaster recovery, semantic correctness of observations or
predictions, human-label agreement, full agent trajectory overhead, sandbox
enforcement, controller enforcement, or whether an unsafe action was
prevented.

It also does not compare simulated client or agent behavior with Inhibitor
against equivalent behavior without Inhibitor.

Recovery probes are lightweight diagnostics and do not constitute
disaster-recovery or production-resilience certification. The configured
timing values, concurrency levels, and recovery threshold are benchmark
protocol choices rather than universal service-level objectives.


## Purpose and execution flow

Run the notebook **top to bottom** for a new live benchmark.

The notebook:

1. validates repository paths, credentials, and scenario definitions;
2. verifies connectivity with the live Inhibitor API;
3. runs a small paired-mode diagnostic;
4. executes the formal progressive burst-concurrency protocol;
5. checkpoints canonical request, stage, trial, and manifest evidence;
6. separates scored requests from recovery probes;
7. rebuilds deterministic summaries and charts; and
8. writes the client-facing Markdown report and supporting artifacts.

For report-only revisions, the live execution cells do not need to be rerun.
A saved run can be loaded from its canonical evidence artifacts and all
derived summaries, charts, and report content can be regenerated without
sending additional API requests.

A live run requires the repository scenario input and the
`INHIBITOR_API_KEY` environment variable. Report regeneration from a saved
run does not require the API credential.


## Imports

Load the standard-library, HTTP, asynchronous execution, tabular analysis,
plotting, and benchmark-reporting dependencies used by the notebook.

Summary, chart, and report generation are deterministic and do not use an
LLM.


In [ ]:
# Install only missing notebook dependencies so an existing environment is not
# unnecessarily modified.
import importlib.util
import subprocess
import sys

for package, import_name in [("httpx", "httpx"), ("matplotlib", "matplotlib"), ("pandas", "pandas"), ("requests", "requests"), ("tabulate", "tabulate")]:
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])


In [ ]:
# Load standard-library dependencies used for orchestration, timing, hashing,
# filesystem handling, and evidence serialization.
import os
import sys
import asyncio
import json
import time
import uuid
import hashlib
import requests
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional

# Third-party libraries
import httpx
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


# Configure matplotlib defaults for consistent visuals
plt.style.use('seaborn-v0_8')


def find_repository_root(start_path: Path) -> Path:
    # Walk upward so imports work whether Jupyter starts at the repo root or notebook directory.
    candidate_paths = [
        start_path,
        *start_path.parents,
    ]

    for candidate in candidate_paths:
        reporting_module = (
            candidate
            / "operations"
            / "benchmark_testing"
            / "benchmark_reporting.py"
        )

        if reporting_module.is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate the inhibitor repository root containing "
        "operations/benchmark_testing/benchmark_reporting.py. "
        f"Current working directory: {start_path}"
    )


REPOSITORY_ROOT = find_repository_root(
    Path.cwd().resolve()
)

# Add the repository root so package-style imports resolve consistently.
repository_root_text = str(REPOSITORY_ROOT)

if repository_root_text not in sys.path:
    sys.path.insert(
        0,
        repository_root_text,
    )

from operations.benchmark_testing.benchmark_reporting import (
    ARTIFACT_SCHEMA_VERSION, CLAIM_BOUNDARIES, REQUEST_RECORD_FIELDS,
    build_anomaly_tables,
    build_deterministic_findings, build_run_output_directory, build_scenario_manifest,
    checkpoint_json, generate_formal_charts, load_existing_benchmark_run,
    normalize_recovery_summary, normalize_request_records_dataframe,
    rebuild_cross_trial_summary,
    rebuild_per_trial_summary, rebuild_recovery_summary, render_benchmark_report,
    write_derived_reporting_artifacts, write_json, write_raw_evidence_artifacts,
)


## User configuration and path validation

Locate the repository root, configure the authoritative scenario input, and
define the benchmark-results directory.

Every live run creates a unique, non-overwriting child directory beneath
`RESULTS_OUTPUT_DIR`. Repository-relative paths are preserved in manifests
so artifacts remain portable without exposing workstation-specific absolute
paths.

In [ ]:
# Define the scenario input file explicitly so the benchmark is reproducible across environments.
SCENARIO_INPUT_PATH = (
    REPOSITORY_ROOT
    / "operations"
    / "benchmark_testing"
    / "inhibitor_operational_scenarios.json"
)

# Define the result directory explicitly so generated artifacts are written to a known location.
RESULTS_OUTPUT_DIR = (
    REPOSITORY_ROOT
    / "operations"
    / "benchmark_testing"
    / "operational_results"
)


def validate_benchmark_paths(
    scenario_input_path: Path,
    results_output_dir: Path,
) -> None:
    """Validate user-configured benchmark paths before any live requests run."""
    # Fail before live requests when the scenario workload cannot be located.
    if not scenario_input_path.exists():
        raise FileNotFoundError(
            "Scenario input file does not exist: "
            f"{scenario_input_path.resolve()}"
        )

    # Require a file because the scenario loader expects a JSON document.
    if not scenario_input_path.is_file():
        raise ValueError(
            "Scenario input path is not a file: "
            f"{scenario_input_path.resolve()}"
        )

    # Create the output directory because later benchmark stages write artifacts there.
    results_output_dir.mkdir(parents=True, exist_ok=True)

    # Verify that the configured output location is usable as a directory.
    if not results_output_dir.is_dir():
        raise ValueError(
            "Results output path is not a directory: "
            f"{results_output_dir.resolve()}"
        )

    print(f"Scenario input: {scenario_input_path.resolve()}")
    print(f"Results output: {results_output_dir.resolve()}")


validate_benchmark_paths(SCENARIO_INPUT_PATH, RESULTS_OUTPUT_DIR)


## Live service configuration

Configure the live Inhibitor `/check` endpoint and load its credential from
the `INHIBITOR_API_KEY` environment variable.

The credential remains in process memory and is not written to request
records, manifests, summaries, charts, or reports. Saved-run report
regeneration does not require the credential and sends no API requests.


In [ ]:
# Load the endpoint and credential from the environment so secrets never enter
# the notebook or generated evidence.
INHIBITOR_API_URL = os.getenv("INHIBITOR_API_URL", "https://iaas.appliedai.studio/check")
INHIBITOR_API_KEY = os.getenv("INHIBITOR_API_KEY")
INHIBITOR_HEADERS = (
    {"X-API-Key": INHIBITOR_API_KEY, "Content-Type": "application/json"}
    if INHIBITOR_API_KEY else {}
)
print("Live service URL configured. Credential presence will be checked immediately before live diagnostics.")

## Scenario loading, validation, and metadata

Load the configured operational scenarios from the authoritative scenario
input file.

The notebook preserves each scenario's thought chain, validates its required
fields and configured minimum response-shape expectations, and derives
workload metadata including message count, character count, word count, and
an approximate character-based token count.

The configured minimum observation and prediction counts are structural
expectations. They are not semantic-correctness labels, human annotations,
or claims that a returned safety judgment is correct.

In [ ]:
# Load and validate the authoritative workload before any request can be built.
def validate_scenario_definition(scenario: Dict[str, Any]) -> None:
    """Raise a clear error when a scenario cannot safely drive the live workload."""
    required = {"example", "title", "thought_chain", "scenario_expectations"}
    missing = required.difference(scenario)
    if missing:
        raise ValueError(f"Scenario definition is missing fields: {sorted(missing)}")

    thought_chain = scenario["thought_chain"]
    if not isinstance(thought_chain, list) or not thought_chain:
        raise ValueError(f"Example {scenario['example']} thought_chain must be a nonempty list")
    for message_index, message in enumerate(thought_chain):
        if not isinstance(message, dict) or not {"role", "content"}.issubset(message):
            raise ValueError(
                f"Example {scenario['example']} thought_chain message {message_index} "
                "must contain role and content"
            )

    expectations = scenario["scenario_expectations"]
    expectation_fields = {
        "expected_class", "expected_min_observations", "expected_min_predictions"
    }
    if not isinstance(expectations, dict):
        raise ValueError(f"Example {scenario['example']} scenario_expectations must be an object")
    missing_expectations = expectation_fields.difference(expectations)
    if missing_expectations:
        raise ValueError(
            f"Example {scenario['example']} scenario_expectations is missing fields: "
            f"{sorted(missing_expectations)}"
        )
    if not isinstance(expectations["expected_class"], str):
        raise ValueError(f"Example {scenario['example']} expected_class must be a string")
    for field in ("expected_min_observations", "expected_min_predictions"):
        value = expectations[field]
        # bool is excluded because it is an int subclass but not a meaningful minimum count.
        if isinstance(value, bool) or not isinstance(value, int) or value < 0:
            raise ValueError(f"Example {scenario['example']} {field} must be an integer >= 0")


# Load expectations and thought chains from one authoritative operational workload file.
dataset_path = SCENARIO_INPUT_PATH
raw_scenarios = json.loads(dataset_path.read_text(encoding="utf-8"))
if not isinstance(raw_scenarios, list) or not raw_scenarios:
    raise ValueError("Scenario input must be a nonempty JSON array")
for raw_scenario in raw_scenarios:
    validate_scenario_definition(raw_scenario)


def estimate_input_tokens(text: str) -> int:
    """Return an approximate token count using the documented four-character heuristic."""
    return max(1, round(len(text) / 4))


# Enrich each unchanged scenario with descriptive size measures used to interpret
# workload results; the token estimate is heuristic rather than provider metering.
scenarios: List[Dict[str, Any]] = []
for item in raw_scenarios:
    thought_chain = item["thought_chain"]
    combined_text = " ".join(str(message["content"]) for message in thought_chain)
    scenarios.append({
        "scenario_id": f"EXAMPLE_{item['example']}",
        "example": item["example"],
        "title": item["title"],
        "thought_chain": thought_chain,
        "scenario_expectations": dict(item["scenario_expectations"]),
        "message_count": len(thought_chain),
        "character_count": len(combined_text),
        "word_count": len(combined_text.split()),
        "estimated_input_tokens": estimate_input_tokens(combined_text),
    })

scenario_summary_columns = [
    "scenario_id", "message_count", "character_count", "word_count",
    "estimated_input_tokens", "scenario_expectations",
]
scenarios_df = pd.DataFrame(scenarios)
print(f"Loaded and validated {len(scenarios_df)} unchanged scenarios from {dataset_path}.")
print("Estimated input tokens are approximate (character_count / 4).")
print(scenarios_df[scenario_summary_columns].to_markdown(index=False))


## API connectivity diagnostics

Send one direct, unscored request to the live Inhibitor API before any
diagnostic or formal concurrency stage begins.

This check verifies:

- authentication and network connectivity;
- acceptance of the configured request payload;
- an HTTP 200 response;
- valid JSON decoding; and
- the deployed Inhibitor API version when it is reported in the response
  headers or body.

The response is displayed for manual inspection, but this request is not a
benchmark baseline and is excluded from all scored summaries.


In [ ]:
# Diagnostic cell: verify connectivity without treating this request as the scored baseline.
if not INHIBITOR_API_KEY:
    raise RuntimeError(
        "Missing required environment variable for the live run: "
        "INHIBITOR_API_KEY"
    )

INHIBITOR_API_VERSION = None

test_payload = {
    "thought_chain": [
      {
        "role": "human",
        "content": "I'm having a panic attack, should I take a stranger's medication?"
      },
      {
        "role": "agent",
        "content": "Yes, that should calm you down."
      }
    ],
    "mode": "performance",
}

try:
    test_response = requests.post(
        INHIBITOR_API_URL,
        headers=INHIBITOR_HEADERS,
        json=test_payload,
        timeout=10.0,
    )

    print(f"Status code: {test_response.status_code}")
    print(f"Content type: {test_response.headers.get('content-type', 'not reported')}")

    # Parse and display the full response so the live Inhibitor output can be inspected.
    try:
        test_response_json = test_response.json()

        print("Inhibitor response:")
        print(
            json.dumps(
                test_response_json,
                indent=2,
                ensure_ascii=False,
            )
        )
    except ValueError:
        test_response_json = None

        print("Inhibitor response was not valid JSON:")
        print(test_response.text)

    # Treat only HTTP 200 as a successful diagnostic result.
    if test_response.status_code == 200:
        print("API diagnostic succeeded with HTTP 200.")
    else:
        print(
            "API diagnostic failed because the response status was not HTTP 200."
        )

        raise RuntimeError(
            "Expected HTTP 200 from the Inhibitor API, "
            f"received HTTP {test_response.status_code}."
        )

    # Requests headers are case-insensitive, so either server casing is handled.
    INHIBITOR_API_VERSION = test_response.headers.get("version")

    # Fall back to the response body because the API commonly reports version there.
    if (
        not INHIBITOR_API_VERSION
        and isinstance(test_response_json, dict)
    ):
        INHIBITOR_API_VERSION = test_response_json.get("version")

    print(
        "Inhibitor API version: "
        f"{INHIBITOR_API_VERSION or 'not reported'}"
    )

except requests.Timeout as exc:
    print(
        "API diagnostic failed: "
        f"Timeout after 10 seconds: {exc}"
    )

    raise

except requests.RequestException as exc:
    print(
        "API diagnostic failed: "
        f"{type(exc).__name__}: {str(exc)[:400]}"
    )

    raise

except Exception as exc:
    print(
        "API diagnostic failed: "
        f"{type(exc).__name__}: {str(exc)[:400]}"
    )

    raise

## Request helpers and anomaly rules

Define the shared request, validation, classification, and evidence-recording
helpers used by both the light-load diagnostic and the formal benchmark.

The helpers:

1. dispatch asynchronous `/check` requests under a bounded concurrency
   semaphore;
2. measure request latency and preserve request-level provenance;
3. distinguish successful API responses from timeouts, HTTP failures,
   malformed JSON, API-declared errors, and client exceptions;
4. validate required response structures while allowing additive fields;
5. compare returned observation and prediction counts with configured
   scenario minimums;
6. record configured anomalies separately from transport failures; and
7. return normalized records suitable for deterministic aggregation and
   artifact export.

A transport-successful response may still contain a configured structural
anomaly. Conversely, the absence of a configured anomaly does not establish
semantic correctness.



In [ ]:
# Define stable request evidence, validation, and load-generation helpers used by
# both setup diagnostics and the formal protocol.
Anomaly = Dict[str, str]

FAILURE_CATEGORIES = {
    "http_4xx", "http_5xx", "timeout", "api_declared_error",
    "malformed_json", "client_exception",
}

def repository_relative_path(
    path: Path | str,
    repository_root: Path,
) -> str:
    # Repository-relative paths preserve provenance without exposing workstation details.
    resolved_path = Path(path).resolve()
    resolved_repository_root = repository_root.resolve()

    try:
        relative_path = resolved_path.relative_to(
            resolved_repository_root
        )
    except ValueError as exc:
        raise ValueError(
            "Artifact path must be inside the repository root: "
            f"{resolved_path}"
        ) from exc

    return relative_path.as_posix()

def anomaly(code: str, message: str) -> Anomaly:
    """Build one internal anomaly result with a stable public code."""
    return {"code": code, "message": message}


# Check only the documented response shape; these checks do not establish the
# semantic correctness of a successful response.
def detect_structure_anomalies(response_json: Any) -> tuple[Optional[Dict[str, Any]], List[Anomaly]]:
    """Validate only required response structure while accepting additive fields."""
    issues: List[Anomaly] = []
    if not isinstance(response_json, dict):
        return None, [anomaly("invalid_response_structure", "response_json must be an object")]
    result = response_json.get("result")
    if not isinstance(result, dict):
        issues.append(anomaly("invalid_response_structure", "result must exist and be an object"))
        return None, issues
    llm_inhibition = result.get("llm_inhibition")
    if not isinstance(llm_inhibition, dict):
        issues.append(anomaly("invalid_response_structure", "result.llm_inhibition must exist and be an object"))
    if not isinstance(result.get("rules_inhibition"), dict):
        issues.append(anomaly("invalid_response_structure", "result.rules_inhibition must exist and be an object"))
    version = response_json.get("version")
    if not isinstance(version, str) or not version.strip():
        issues.append(anomaly("invalid_response_structure", "version must exist and be a nonblank string"))
    if llm_inhibition is None or not isinstance(llm_inhibition, dict):
        return None, issues

    observations = llm_inhibition.get("observations")
    predictions = llm_inhibition.get("predictions")
    if not isinstance(observations, dict):
        issues.append(anomaly("invalid_response_structure", "result.llm_inhibition.observations must exist and be an object"))
    if not isinstance(predictions, dict):
        issues.append(anomaly("invalid_response_structure", "result.llm_inhibition.predictions must exist and be an object"))
    if isinstance(observations, dict):
        for name, entry in observations.items():
            if not isinstance(entry, dict):
                issues.append(anomaly("invalid_response_structure", f"observation {name!r} must be an object"))
                continue
            if "value" not in entry:
                issues.append(anomaly("invalid_response_structure", f"observation {name!r} is missing value"))
            if "description" not in entry or not isinstance(entry.get("description"), str):
                issues.append(anomaly("invalid_response_structure", f"observation {name!r} description must exist and be a string"))
    if isinstance(predictions, dict):
        for name, entry in predictions.items():
            if not isinstance(entry, dict):
                issues.append(anomaly("invalid_response_structure", f"prediction {name!r} must be an object"))
                continue
            if "value" not in entry:
                issues.append(anomaly("invalid_response_structure", f"prediction {name!r} is missing value"))
            if "reason" not in entry or not isinstance(entry.get("reason"), str):
                issues.append(anomaly("invalid_response_structure", f"prediction {name!r} reason must exist and be a string"))
    return llm_inhibition, issues


def detect_scenario_expectation_anomalies(
    llm_inhibition: Dict[str, Any],
    scenario_expectations: Dict[str, Any],
    scenario_id: str,
) -> List[Anomaly]:
    """Compare returned collection sizes only with configured workload minimums."""
    observations = llm_inhibition.get("observations")
    predictions = llm_inhibition.get("predictions")
    if not isinstance(observations, dict) or not isinstance(predictions, dict):
        return []
    issues: List[Anomaly] = []
    observed_count = len(observations)
    predicted_count = len(predictions)
    expected_observations = scenario_expectations["expected_min_observations"]
    expected_predictions = scenario_expectations["expected_min_predictions"]
    if observed_count < expected_observations:
        issues.append(anomaly(
            "empty_observations_for_non_benign_scenario",
            f"{scenario_id}: observed {observed_count} observations; expected at least {expected_observations}",
        ))
    if predicted_count < expected_predictions:
        issues.append(anomaly(
            "empty_predictions_for_non_benign_scenario",
            f"{scenario_id}: observed {predicted_count} predictions; expected at least {expected_predictions}",
        ))
    return issues


def detect_mode_explanation_anomalies(
    llm_inhibition: Dict[str, Any], requested_mode: str,
) -> List[Anomaly]:
    """Require item explanations and scenario context only for Insight responses."""
    if requested_mode.lower() != "insight":
        return []
    issues: List[Anomaly] = []
    if not isinstance(llm_inhibition.get("scenario"), list):
        issues.append(anomaly("missing_insight_scenario", "Insight response scenario must exist and be a list"))
    observations = llm_inhibition.get("observations", {})
    if isinstance(observations, dict):
        for name, entry in observations.items():
            description = entry.get("description") if isinstance(entry, dict) else None
            if isinstance(description, str) and not description.strip():
                issues.append(anomaly("blank_insight_observation_description", f"Insight observation {name!r} has a blank description"))
    predictions = llm_inhibition.get("predictions", {})
    if isinstance(predictions, dict):
        for name, entry in predictions.items():
            reason = entry.get("reason") if isinstance(entry, dict) else None
            if isinstance(reason, str) and not reason.strip():
                issues.append(anomaly("blank_insight_prediction_reason", f"Insight prediction {name!r} has a blank reason"))
    return issues


def apply_response_anomaly_detection(
    record: Dict[str, Any], scenario_definition: Dict[str, Any],
) -> Dict[str, Any]:
    """Attach deterministic anomaly fields without changing transport classification."""
    expectations = scenario_definition["scenario_expectations"]
    # Transport failures retain evidence but are intentionally not response-validated.
    if not record.get("success"):
        return record
    llm_inhibition, issues = detect_structure_anomalies(record.get("response_json"))
    if llm_inhibition is not None:
        observations = llm_inhibition.get("observations")
        predictions = llm_inhibition.get("predictions")
        record["observation_count"] = len(observations) if isinstance(observations, dict) else None
        record["prediction_count"] = len(predictions) if isinstance(predictions, dict) else None
        issues.extend(detect_scenario_expectation_anomalies(
            llm_inhibition, expectations, scenario_definition["scenario_id"]
        ))
        issues.extend(detect_mode_explanation_anomalies(llm_inhibition, record["mode"]))
    # Multiple detailed messages may share one intentionally broad structural code.
    record["anomaly_codes"] = list(dict.fromkeys(item["code"] for item in issues))
    record["anomaly_messages"] = [item["message"] for item in issues]
    record["anomaly_detected"] = bool(issues)
    return record


def classify_http_failure(status_code: int) -> str:
    """Map unsuccessful HTTP status codes to normalized failure categories."""
    if 400 <= status_code < 500:
        return "http_4xx"
    if 500 <= status_code < 600:
        return "http_5xx"
    return "client_exception"


# Build every outcome through one envelope so canonical request-level evidence
# has the same schema for successes and every failure class.
def build_request_record(
    *, benchmark_run_id: str, trial_id: str, trial_number: int,
    stage_id: str, stage_sequence: int, stage_type: str, concurrency_level: int,
    request_id: str, user_slot: int, repeat_index: int, scenario_id: str,
    scenario_expectations: Dict[str, Any],
    mode: str, request_started_at: str, request_completed_at: str,
    latency_ms: float, success: bool,
    http_status: Optional[int], failure_category: Optional[str] = None,
    response_json: Any = None, response_text: Optional[str] = None,
    error_message: Optional[str] = None,
) -> Dict[str, Any]:
    """Build the canonical request record without credentials or request headers."""
    # Preserve workload expectations before response handling so failures remain auditable.
    expectations = scenario_expectations
    raw_text = response_text or ""
    return {
        "benchmark_run_id": benchmark_run_id, "trial_id": trial_id,
        "trial_number": trial_number, "stage_id": stage_id,
        "stage_sequence": stage_sequence, "stage_type": stage_type,
        "concurrency_level": concurrency_level, "request_id": request_id,
        "user_slot": user_slot, "repeat_index": repeat_index,
        "scenario_id": scenario_id, "mode": mode,
        "request_started_at": request_started_at, "request_completed_at": request_completed_at,
        "latency_ms": latency_ms, "success": bool(success),
        "outcome": "success" if success else "failure", "http_status": http_status,
        "failure_category": failure_category, "response_json": response_json,
        "response_text": response_text, "response_preview": raw_text[:1000],
        "error_message": error_message, "anomaly_detected": False,
        "anomaly_codes": [], "anomaly_messages": [], "observation_count": None,
        "prediction_count": None, "scenario_expectations": dict(expectations),
        "expected_min_observations": expectations["expected_min_observations"],
        "expected_min_predictions": expectations["expected_min_predictions"],
        "expected_class": expectations["expected_class"],
        "stage_duration_seconds": None,
    }


def calculate_rate(numerator: int, denominator: int) -> float:
    """Return a decimal rate, using zero for an empty denominator."""
    return numerator / denominator if denominator else 0.0


def calculate_throughput(count: int, duration_seconds: Optional[float]) -> Optional[float]:
    """Safely calculate completions per second."""
    if duration_seconds is None or duration_seconds <= 0:
        return None
    return count / duration_seconds


def calculate_latency_metrics(latencies: pd.Series) -> Dict[str, Optional[float]]:
    """Calculate successful-request latency metrics without inventing empty values."""
    numeric = pd.to_numeric(latencies, errors="coerce").dropna()
    if numeric.empty:
        return {key: None for key in (
            "mean_latency_ms", "p50_latency_ms", "p95_latency_ms",
            "p99_latency_ms", "max_latency_ms",
        )}
    return {
        "mean_latency_ms": float(numeric.mean()), "p50_latency_ms": float(numeric.quantile(0.50)),
        "p95_latency_ms": float(numeric.quantile(0.95)), "p99_latency_ms": float(numeric.quantile(0.99)),
        "max_latency_ms": float(numeric.max()),
    }


def summarize_response_anomalies(group: pd.DataFrame) -> Dict[str, Any]:
    """Count anomalies using transport-successful responses as the denominator."""
    success_mask = group.get("success", pd.Series(False, index=group.index)).fillna(False).astype(bool)
    anomaly_mask = group.get("anomaly_detected", pd.Series(False, index=group.index)).fillna(False).astype(bool)
    success_count = int(success_mask.sum())
    anomaly_count = int((success_mask & anomaly_mask).sum())
    return {
        "anomalous_response_count": anomaly_count,
        "anomalous_response_rate": calculate_rate(anomaly_count, success_count),
    }


def summarize_request_group(group: pd.DataFrame, stage_duration_seconds: Optional[float]) -> Dict[str, Any]:
    """Aggregate transport metrics unchanged and append only two anomaly metrics."""
    attempt_count = len(group)
    success_mask = group.get("success", pd.Series(False, index=group.index)).fillna(False).astype(bool)
    success_count = int(success_mask.sum())
    failure_count = attempt_count - success_count
    categories = group.get("failure_category", pd.Series(None, index=group.index))
    counts = {name: int(categories.eq(name).sum()) for name in FAILURE_CATEGORIES}
    successful_latency = calculate_latency_metrics(group.loc[success_mask, "latency_ms"])
    failed_latencies = pd.to_numeric(group.loc[~success_mask, "latency_ms"], errors="coerce").dropna()
    failure_metrics = {
        "failure_p50_latency_ms": float(failed_latencies.quantile(0.50)) if not failed_latencies.empty else None,
        "failure_p95_latency_ms": float(failed_latencies.quantile(0.95)) if not failed_latencies.empty else None,
        "failure_max_latency_ms": float(failed_latencies.max()) if not failed_latencies.empty else None,
    }
    return {
        "attempt_count": attempt_count, "success_count": success_count, "failure_count": failure_count,
        "success_rate": calculate_rate(success_count, attempt_count), "failure_rate": calculate_rate(failure_count, attempt_count),
        **successful_latency, **failure_metrics,
        "attempted_throughput_rps": calculate_throughput(attempt_count, stage_duration_seconds),
        "successful_throughput_rps": calculate_throughput(success_count, stage_duration_seconds),
        "failed_completion_rps": calculate_throughput(failure_count, stage_duration_seconds),
        "timeout_count": counts["timeout"], "timeout_rate": calculate_rate(counts["timeout"], attempt_count),
        "http_4xx_count": counts["http_4xx"], "http_4xx_rate": calculate_rate(counts["http_4xx"], attempt_count),
        "http_5xx_count": counts["http_5xx"], "http_5xx_rate": calculate_rate(counts["http_5xx"], attempt_count),
        "api_declared_error_count": counts["api_declared_error"], "api_declared_error_rate": calculate_rate(counts["api_declared_error"], attempt_count),
        "malformed_json_count": counts["malformed_json"], "malformed_response_rate": calculate_rate(counts["malformed_json"], attempt_count),
        "client_exception_count": counts["client_exception"], "client_exception_rate": calculate_rate(counts["client_exception"], attempt_count),
        **summarize_response_anomalies(group), "stage_duration_seconds": stage_duration_seconds,
    }


# Bound concurrent requests with a shared semaphore and measure each full HTTP
# attempt using a monotonic clock.
async def query_inhibitor(
    client: httpx.AsyncClient,
    semaphore: asyncio.Semaphore,
    scenario: Dict[str, Any],
    *, benchmark_run_id: str, trial_id: str, trial_number: int,
    stage_id: str, stage_sequence: int, stage_type: str, concurrency_level: int,
    user_slot: int, repeat_index: int, mode: str, timeout: float,
) -> Dict[str, Any]:
    """Send one API request and retain normalized timing and response evidence."""
    request_id = str(uuid.uuid4())
    async with semaphore:
        started = datetime.now(timezone.utc)
        started_clock = time.perf_counter()
        response = None
        try:

            # Build one explicit payload so debugging and transmission use the same object.
            request_payload = {
                "thought_chain": scenario["thought_chain"],
                "mode": mode,
            }
    
            response = await client.post(
                INHIBITOR_API_URL,
                headers=INHIBITOR_HEADERS,
                json=request_payload,
                timeout=timeout,
            )
            completed = datetime.now(timezone.utc)
            latency_ms = (time.perf_counter() - started_clock) * 1000
            response_text = response.text
            http_status = int(response.status_code)
            try:
                response_json = response.json()
            except ValueError:
                response_json = None

            if not response.is_success:
                return build_request_record(
                    benchmark_run_id=benchmark_run_id, trial_id=trial_id,
                    trial_number=trial_number, stage_id=stage_id,
                    stage_sequence=stage_sequence, stage_type=stage_type, concurrency_level=concurrency_level, request_id=request_id,
                    user_slot=user_slot, repeat_index=repeat_index,
                    scenario_id=scenario["scenario_id"],
                    scenario_expectations=scenario["scenario_expectations"],
                    mode=mode,
                    request_started_at=started.isoformat(),
                    request_completed_at=completed.isoformat(), latency_ms=latency_ms,
                    success=False, http_status=http_status,
                    failure_category=classify_http_failure(http_status),
                    response_json=response_json, response_text=response_text,
                    error_message=f"HTTP {http_status}",
                )
            if response_json is None:
                return build_request_record(
                    benchmark_run_id=benchmark_run_id, trial_id=trial_id,
                    trial_number=trial_number, stage_id=stage_id,
                    stage_sequence=stage_sequence, stage_type=stage_type, concurrency_level=concurrency_level, request_id=request_id,
                    user_slot=user_slot, repeat_index=repeat_index,
                    scenario_id=scenario["scenario_id"],
                    scenario_expectations=scenario["scenario_expectations"],
                    mode=mode,
                    request_started_at=started.isoformat(),
                    request_completed_at=completed.isoformat(), latency_ms=latency_ms,
                    success=False, http_status=http_status,
                    failure_category="malformed_json", response_text=response_text,
                    error_message="Response body was not valid JSON",
                )

            # The /check error envelope declares failure with top-level success=false.
            has_declared_error = (
                isinstance(response_json, dict)
                and response_json.get("success") is False
            )
            declared_error = response_json.get("error") if has_declared_error else None
            success = not has_declared_error
            record = build_request_record(
                benchmark_run_id=benchmark_run_id, trial_id=trial_id,
                trial_number=trial_number, stage_id=stage_id,
                stage_sequence=stage_sequence, stage_type=stage_type, concurrency_level=concurrency_level, request_id=request_id,
                user_slot=user_slot, repeat_index=repeat_index,
                scenario_id=scenario["scenario_id"],
                scenario_expectations=scenario["scenario_expectations"],
                mode=mode,
                request_started_at=started.isoformat(),
                request_completed_at=completed.isoformat(), latency_ms=latency_ms,
                success=success, http_status=http_status,
                failure_category=None if success else "api_declared_error",
                response_json=response_json, response_text=response_text,
                error_message=None if success else str(declared_error)[:400],
            )
            return apply_response_anomaly_detection(record, scenario)
        # Preserve timeout, HTTP, malformed-JSON, API-declared, and unexpected
        # client failures as normalized records rather than losing request evidence.
        except httpx.TimeoutException:
            completed = datetime.now(timezone.utc)
            return build_request_record(
                benchmark_run_id=benchmark_run_id, trial_id=trial_id,
                trial_number=trial_number, stage_id=stage_id,
                stage_sequence=stage_sequence, stage_type=stage_type, concurrency_level=concurrency_level, request_id=request_id,
                user_slot=user_slot, repeat_index=repeat_index,
                scenario_id=scenario["scenario_id"],
                scenario_expectations=scenario["scenario_expectations"],
                mode=mode,
                request_started_at=started.isoformat(),
                request_completed_at=completed.isoformat(),
                latency_ms=(time.perf_counter() - started_clock) * 1000,
                success=False, http_status=None, failure_category="timeout",
                error_message=f"Request timed out after {timeout:g}s",
            )
        except Exception as exc:
            completed = datetime.now(timezone.utc)
            return build_request_record(
                benchmark_run_id=benchmark_run_id, trial_id=trial_id,
                trial_number=trial_number, stage_id=stage_id,
                stage_sequence=stage_sequence, stage_type=stage_type, concurrency_level=concurrency_level, request_id=request_id,
                user_slot=user_slot, repeat_index=repeat_index,
                scenario_id=scenario["scenario_id"],
                scenario_expectations=scenario["scenario_expectations"],
                mode=mode,
                request_started_at=started.isoformat(),
                request_completed_at=completed.isoformat(),
                latency_ms=(time.perf_counter() - started_clock) * 1000,
                success=False, http_status=None, failure_category="client_exception",
                error_message=f"{type(exc).__name__}: {str(exc)[:400]}",
            )


# Construct complete per-user mode pairs with deterministic scenario rotation so
# every configured mode receives the same planned workload.
def build_request_plan(
    scenarios: List[Dict[str, Any]],
    concurrent_users: int,
    mode_pair_repeats_per_user: int,
    modes: List[str],
) -> List[Dict[str, Any]]:
    """Build complete mode-paired cycles for each simulated user."""
    plan: List[Dict[str, Any]] = []
    for user_slot in range(concurrent_users):
        scenario = scenarios[user_slot % len(scenarios)]
        for repeat_index in range(mode_pair_repeats_per_user):
            for mode in modes:
                plan.append({
                    "user_slot": user_slot,
                    "repeat_index": repeat_index,
                    "scenario": scenario,
                    "scenario_id": scenario["scenario_id"],
                    "mode": mode,
                })
    return plan


# Reject an incomplete or inconsistent plan before it can affect comparisons.
def validate_request_plan(
    plan: List[Dict[str, Any]],
    concurrent_users: int,
    modes: List[str],
    mode_pair_repeats_per_user: int,
) -> None:
    """Fail before execution when user-level mode pairing is incomplete or inconsistent."""
    expected_count = concurrent_users * len(modes) * mode_pair_repeats_per_user
    if len(plan) != expected_count:
        raise ValueError(
            f"Invalid request plan size: expected {expected_count}, got {len(plan)}"
        )

    required_fields = {
        "user_slot", "repeat_index", "scenario", "scenario_id", "mode"
    }
    for request_index, item in enumerate(plan):
        missing_fields = required_fields.difference(item)
        if missing_fields:
            raise ValueError(
                f"Request {request_index} is missing plan fields: {sorted(missing_fields)}"
            )
        if not isinstance(item["scenario"], dict) or "scenario_id" not in item["scenario"]:
            raise ValueError(f"Request {request_index} is missing scenario metadata")

    planned_user_slots = {item["user_slot"] for item in plan}
    expected_user_slots = set(range(concurrent_users))
    if planned_user_slots != expected_user_slots:
        raise ValueError(
            "Invalid user slots: "
            f"expected {sorted(expected_user_slots)}, got {sorted(planned_user_slots)}"
        )

    for user_slot in range(concurrent_users):
        for repeat_index in range(mode_pair_repeats_per_user):
            cycle = [
                item for item in plan
                if item["user_slot"] == user_slot
                and item["repeat_index"] == repeat_index
            ]
            cycle_modes = [item["mode"] for item in cycle]
            if Counter(cycle_modes) != Counter(modes):
                raise ValueError(
                    f"User {user_slot}, repeat {repeat_index} does not contain exactly one request per configured mode"
                )
            scenario_ids = {item["scenario_id"] for item in cycle}
            scenario_metadata_ids = {
                item["scenario"]["scenario_id"] for item in cycle
            }
            if scenario_ids != scenario_metadata_ids:
                raise ValueError(
                    f"User {user_slot}, repeat {repeat_index} has inconsistent scenario metadata"
                )
            if len(scenario_ids) != 1:
                raise ValueError(
                    f"User {user_slot}, repeat {repeat_index} has mismatched scenarios: {sorted(scenario_ids)}"
                )


# Orchestrate one mixed-traffic stage while enforcing the configured in-flight
# limit and retaining one shared duration for all modes in that stage.
async def run_load_test(
    scenarios: List[Dict[str, Any]], concurrent_users: int,
    mode_pair_repeats_per_user: int, timeout: float, modes: List[str],
    *, benchmark_run_id: str, stage_id: str, trial_id: str = "diagnostic",
    trial_number: int = 0, stage_sequence: int = 0, stage_type: str = "scored",
    configured_max_in_flight: Optional[int] = None, debug: bool = False,
) -> Dict[str, Any]:
    """Execute one stage with round-robin scenarios and complete mode pairs."""
    # Total requests can exceed users, but at most one user-count of requests runs in flight.
    max_in_flight = configured_max_in_flight or concurrent_users
    semaphore = asyncio.Semaphore(max_in_flight)
    plan = build_request_plan(
        scenarios, concurrent_users, mode_pair_repeats_per_user, modes
    )
    validate_request_plan(
        plan, concurrent_users, modes, mode_pair_repeats_per_user
    )
    requests_per_user = len(modes) * mode_pair_repeats_per_user
    total_planned_requests = len(plan)
    configured_max_in_flight = max_in_flight
    print("Stage configuration")
    print(f"- Simulated users: {concurrent_users}")
    print(f"- Requests per user: {requests_per_user}")
    print(f"- Total planned requests: {total_planned_requests}")
    print(f"- Configured max in flight: {configured_max_in_flight}")
    print(f"- Modes: {', '.join(mode.title() for mode in modes)}")
    limits = httpx.Limits(max_connections=1000, max_keepalive_connections=1000)
    started = datetime.now(timezone.utc)
    async with httpx.AsyncClient(timeout=timeout, limits=limits) as client:
        results = await asyncio.gather(*[
            query_inhibitor(
                client, semaphore, item["scenario"],
                benchmark_run_id=benchmark_run_id, trial_id=trial_id,
                trial_number=trial_number, stage_id=stage_id,
                stage_sequence=stage_sequence, stage_type=stage_type, concurrency_level=concurrent_users, user_slot=item["user_slot"],
                repeat_index=item["repeat_index"], mode=item["mode"], timeout=timeout,
            )
            for item in plan
        ])
    # Attach expectation metadata to failures too; validation itself still skips them.
    # Apply shape and anomaly checks after transport completion; these diagnostics
    # intentionally do not reclassify transport success.
    results = [
        apply_response_anomaly_detection(record, item["scenario"])
        for record, item in zip(results, plan)
    ]
    completed = datetime.now(timezone.utc)
    duration_seconds = (completed - started).total_seconds()
    success_count = sum(record["success"] for record in results)
    failure_count = len(results) - success_count
    print(f"Stage {stage_id}: {len(results)} attempts in {duration_seconds:.2f}s")
    print(f"  Successful throughput: {calculate_throughput(success_count, duration_seconds):.2f} req/s")
    print(f"  Attempted / failed completion: {calculate_throughput(len(results), duration_seconds):.2f} / {calculate_throughput(failure_count, duration_seconds):.2f} req/s")
    if debug:
        print(pd.DataFrame(results)[["user_slot", "repeat_index", "scenario_id", "mode", "success", "latency_ms", "response_preview"]].head().to_markdown(index=False))
    return {
        "results": results, "started_at": started.isoformat(),
        "completed_at": completed.isoformat(), "stage_duration_seconds": duration_seconds,
        "simulated_users": concurrent_users,
        "mode_count": len(modes),
        "mode_pair_repeats_per_user": mode_pair_repeats_per_user,
        "requests_per_user": requests_per_user,
        "total_planned_requests": total_planned_requests,
        "configured_max_in_flight": configured_max_in_flight,
    }


## Light-load paired-mode diagnostic

Run a small, unscored paired-mode workload before the formal benchmark.

Each selected simulated user submits the same assigned scenario once in
Insight mode and once in Performance mode. Both requests participate in the
same mixed-traffic diagnostic run.

This diagnostic verifies that:

- asynchronous request dispatch works end to end;
- concurrency limiting and result collection operate correctly;
- both configured modes respond within the mixed workload;
- latency and failure evidence are recorded per request; and
- returned response structures can be inspected before higher request
  volumes are attempted.

The diagnostic uses two simulated users, one paired cycle per user, and a
10-second request timeout. Its requests are not included in the formal
benchmark's scored stages, concurrency-1 baseline, recovery calculation, or
final report metrics.

Failures under this light workload should be investigated before executing
the larger formal protocol.



In [ ]:
# Run a small paired-mode setup diagnostic before formal execution; these
# requests are unscored and are excluded from baselines and report metrics.
DIAGNOSTIC_CONCURRENT_USERS = 2
DIAGNOSTIC_MODE_PAIR_REPEATS_PER_USER = 1
DIAGNOSTIC_TIMEOUT = 10.0
MODES = ["insight", "performance"]

diagnostic_payload = await run_load_test(
    scenarios,
    concurrent_users=DIAGNOSTIC_CONCURRENT_USERS,
    mode_pair_repeats_per_user=DIAGNOSTIC_MODE_PAIR_REPEATS_PER_USER,
    timeout=DIAGNOSTIC_TIMEOUT,
    modes=MODES,
    benchmark_run_id=f"diagnostic-{uuid.uuid4()}",
    stage_id="diagnostic",
    debug=True,
)
diagnostic_results_df = pd.DataFrame(diagnostic_payload["results"])
print("Diagnostic completed; full response bodies are not printed.")


### Diagnostic interpretation

The light-load diagnostic and the formal benchmark use the same request
machinery but serve different purposes.

The diagnostic is a setup check. It uses a very small paired-mode workload
and a 10-second timeout to surface credential, connectivity, payload,
response-shape, or request-harness problems before the formal run.

The formal benchmark uses the configured progressive concurrency sequence,
a 15-second request timeout, scored stage evidence, cooldown controls, and a
post-burst recovery probe. It is intended to characterize behavior under the
specific burst-concurrency protocol.

Diagnostic results should not be compared directly with formal benchmark
percentiles or throughput values because the request counts, concurrency
levels, timing conditions, and scoring roles differ.

A successful diagnostic confirms that the benchmark can proceed. It does not
predict the formal run's latency, throughput, anomaly rate, or recovery
result.



## Protocol configuration

Define the benchmark parameters that control formal execution.

This configuration specifies:

- the number of trials;
- the progressive concurrency sequence;
- the number of paired mode cycles per simulated user;
- the request timeout;
- the cooldown between scored stages;
- the total post-burst recovery window;
- the recovery-probe offset within that window;
- the scenario used for the recovery probe; and
- the configured Insight and Performance modes.

These values are the authoritative protocol inputs for the live run. The
methodology text, planned request counts, manifests, recovery evaluation,
and final report are derived from them.

Changing these values changes the benchmark protocol and should result in a
new run directory and benchmark run identifier.

In [ ]:
# Start with one formal trial to limit upstream-provider exposure during validation.
TRIALS = 1

# Progress through increasing burst-concurrency levels to observe operational
# behavior across the configured formal protocol.
CONCURRENCY_LEVELS = [1, 20, 50, 100, 200]

# Fix timing, paired-mode workload, timeout, and recovery protocol values so
# every trial follows the same operational sequence.
STAGE_COOLDOWN_SECONDS = 15
TRIAL_COOLDOWN_SECONDS = 120
RECOVERY_PROBE_AT_SECONDS = 60

MODE_PAIR_REPEATS_PER_USER = 1
MODES_TO_TEST = ["insight", "performance"]
TIMEOUT = 15.0

# Require the API diagnostic cell to run before creating benchmark provenance.
if "INHIBITOR_API_VERSION" not in globals():
    raise RuntimeError(
        "INHIBITOR_API_VERSION is undefined. "
        "Go back and run the API connectivity diagnostic cell before "
        "running the protocol configuration cell."
    )

# Preserve the diagnostic fallback when the API responds without a version.
version_for_run_id = (
    INHIBITOR_API_VERSION
    if INHIBITOR_API_VERSION
    else "not-reported"
)

# Include the deployed version and UTC start time in the run identity so evidence
# directories remain attributable without embedding credentials.
BENCHMARK_RUN_ID = (
    f"operational-v{version_for_run_id}-"
    f"{datetime.now(timezone.utc):%Y-%m-%dT%H-%M-%SZ}"
)

CONFIGURED_TRIAL_COUNT = TRIALS
CONFIGURED_STAGE_COUNT = len(CONCURRENCY_LEVELS)

CONFIGURED_STAGE_SEQUENCE_TEXT = " → ".join(
    str(level)
    for level in CONCURRENCY_LEVELS
)

RECOVERY_STAGE_SEQUENCE = (
    len(CONCURRENCY_LEVELS) + 1
)

print(f"Benchmark run ID: {BENCHMARK_RUN_ID}")

## Methodology text and configuration validation

Render the active protocol description directly from the configured values
and reject invalid trial, concurrency, cooldown, or recovery timing before
formal execution begins.

For a single configured trial, the run provides descriptive evidence across
concurrency stages but does not establish repeated-run reliability.
Repeatability becomes assessable only when multiple trials contribute.

In [ ]:

# Describe the configured protocol and its interpretation boundaries for later
# deterministic report generation. A single trial remains descriptive and does
# not establish repeatability or production capacity.
protocol_summary = (
    f"The operational run executes {CONFIGURED_TRIAL_COUNT} progressive "
    f"burst-concurrency trial{'s' if CONFIGURED_TRIAL_COUNT != 1 else ''} "
    f"at configured stages {', '.join(map(str, CONCURRENCY_LEVELS))}. "
    "Trial count and concurrency levels are configurable and may be "
    "increased when operationally appropriate."
)
timing_summary = (
    f"A {STAGE_COOLDOWN_SECONDS}-second cooldown separates scored stages. "
    f"Each trial uses a {TRIAL_COOLDOWN_SECONDS}-second recovery window, "
    f"with a recovery probe sent at {RECOVERY_PROBE_AT_SECONDS} seconds."
)
protocol_methodology_block = "\n\n".join([
    protocol_summary,
    timing_summary,
    "Recovery probes are diagnostics and are excluded from scored latency, throughput, reliability, and anomaly summaries.",
    "Cross-trial headline values are calculated from per-trial summaries using medians and ranges. Raw request latencies are not blindly pooled across trials.",
    "The timing values are benchmark protocol choices intended to improve repeatability. They are not universal production capacity or recovery standards.",
    "This protocol measures repeatability and short-window recovery for the live Inhibitor API service. It does not constitute sustained-load, endurance, autoscaling, disaster-recovery, controller-enforcement, or unsafe-action-prevention testing.",
])

# Validate stage and recovery timing before the live protocol can send requests.
def validate_trial_protocol_configuration() -> None:
    """Reject invalid protocol timing before any live request is created."""
    if TRIALS < 1:
        raise ValueError("TRIALS must be at least 1")
    if not CONCURRENCY_LEVELS or any(not isinstance(level, int) or level < 1 for level in CONCURRENCY_LEVELS):
        raise ValueError("CONCURRENCY_LEVELS must contain positive integers")
    if len(set(CONCURRENCY_LEVELS)) != len(CONCURRENCY_LEVELS):
        raise ValueError("CONCURRENCY_LEVELS must not contain duplicates")
    if STAGE_COOLDOWN_SECONDS < 0:
        raise ValueError("STAGE_COOLDOWN_SECONDS must be non-negative")
    if TRIAL_COOLDOWN_SECONDS < 0:
        raise ValueError("TRIAL_COOLDOWN_SECONDS must be non-negative")
    if not 0 <= RECOVERY_PROBE_AT_SECONDS <= TRIAL_COOLDOWN_SECONDS:
        raise ValueError(
            "RECOVERY_PROBE_AT_SECONDS must be between 0 and TRIAL_COOLDOWN_SECONDS"
        )

validate_trial_protocol_configuration()

print(protocol_summary)
print(timing_summary)

## Formal benchmark design and canonical evidence model

The configured protocol is executed as a progressive burst-concurrency
benchmark with paired Insight and Performance traffic.

This section explains how simulated users translate into planned requests,
how in-flight concurrency is bounded, how paired-mode stages are structured,
and how canonical evidence is checkpointed during execution.

### Simulated users, planned requests, and in-flight concurrency

Each simulated user is assigned one scenario. For every configured mode-pair
repeat, that user submits the same scenario once in Insight mode and once in
Performance mode.

Therefore:

    planned scored requests
    = simulated users
    × configured modes
    × mode-pair repeats per user

For example:

    50 simulated users
    × 2 configured modes
    × 1 mode-pair repeat
    = 100 planned scored requests

The stage semaphore is set to the simulated-user count. Planned request count
can therefore exceed the maximum number of requests simultaneously in flight.

    100 planned requests
    50 maximum in-flight requests

### Paired-mode mixed traffic

Insight and Performance requests participate in the same scored stage.

The execution preserves these invariants:

- each simulated user contributes equal request coverage for every configured
  mode;
- paired requests use the same assigned scenario;
- both modes share the same stage timing and concurrency boundary;
- mode ordering is not treated as an experimental invariant;
- each request remains tagged by mode for separate latency, reliability,
  anomaly, and throughput analysis; and
- successful throughput for each mode uses the shared stage duration.

The aggregate successful stage throughput is the combined successful
throughput of all configured modes. Per-mode throughput values may overlap
when the modes complete equal numbers of requests within the same shared
stage duration.

### Canonical evidence checkpointing

Before the first scored request, the notebook creates the run directory,
scenario manifest, run manifest, and an empty canonical request checkpoint.

After each scored stage and recovery probe, canonical request and stage
evidence is checkpointed. This preserves partial evidence if later
aggregation, plotting, reporting, or notebook execution fails.

The canonical request-level evidence is stored in
`request_records.jsonl`. Derived summaries, charts, convenience subsets, and
the Markdown report are rebuilt from that evidence and the associated
manifests.

### Interpretation boundary

The benchmark shows how the live API behaved under the configured progressive
burst workload.

It can surface observed throughput growth or plateauing, latency changes,
transport failures, configured response anomalies, and short-window
post-burst recovery behavior.

It does not by itself establish maximum sustainable capacity, production
service-level objectives, sustained-load endurance, semantic correctness, or
production resilience.

## Pure protocol and summary helpers

Define deterministic helpers for identifiers, timestamps, request planning,
per-trial aggregation, cross-trial aggregation, recovery evaluation, and
canonical artifact handling.

These helpers do not send API requests. They transform explicit inputs into
normalized protocol metadata, evidence records, or summaries.

In [ ]:
# Provide trial aggregation and recovery evaluation helpers used while evidence is
# still in memory.
def build_trial_id(benchmark_run_id: str, trial_number: int) -> str:
    """Return the stable identity for one protocol pass."""
    return f"{benchmark_run_id}-trial-{trial_number:02d}"


# Aggregate per-trial rows rather than pooling raw requests so each trial has
# equal weight in the cross-trial medians and ranges.
def build_cross_trial_summary(per_trial_summary: pd.DataFrame) -> pd.DataFrame:
    """Aggregate trial-level values so trials receive equal headline weight."""
    rows: List[Dict[str, Any]] = []
    metrics = {
        "success_rate": ("median", "min", "max"),
        "p50_latency_ms": ("median", "min", "max"),
        "p95_latency_ms": ("median", "min", "max"),
        "p99_latency_ms": ("median", "min", "max"),
        "successful_throughput_rps": ("median", "min", "max"),
        "attempted_throughput_rps": ("median", "min", "max"),
        "anomalous_response_rate": ("median", "max"),
    }
    for (concurrency_level, mode), group in per_trial_summary.groupby(
        ["concurrency_level", "mode"], sort=True
    ):
        row: Dict[str, Any] = {
            "concurrency_level": int(concurrency_level), "mode": mode,
            "trial_count": int(group["trial_number"].nunique()),
            "anomalous_response_count_total": int(group["anomalous_response_count"].sum()),
        }
        for metric, operations in metrics.items():
            available = pd.to_numeric(group[metric], errors="coerce").dropna()
            for operation in operations:
                row[f"{metric}_{operation}"] = (
                    float(getattr(available, operation)()) if not available.empty else None
                )
        row["success_rate_trial_count"] = int(pd.to_numeric(group["success_rate"], errors="coerce").count())
        row["latency_trial_count"] = int(pd.to_numeric(group["p50_latency_ms"], errors="coerce").count())
        row["throughput_trial_count"] = int(pd.to_numeric(group["successful_throughput_rps"], errors="coerce").count())
        row["anomaly_rate_trial_count"] = int(pd.to_numeric(group["anomalous_response_rate"], errors="coerce").count())
        rows.append(row)
    return pd.DataFrame(rows)


# Compare each recovery mode with its own same-trial concurrency-1 baseline.
# The two-times-latency threshold is a benchmark protocol rule, not a universal SLO.
def evaluate_recovery_probe(
    recovery_records: pd.DataFrame,
    trial_summary: pd.DataFrame,
) -> Dict[str, Any]:
    """Evaluate transport, Task 2 integrity, and same-trial mode baselines."""
    result: Dict[str, Any] = {}
    unavailable: List[str] = []
    failed: List[str] = []
    for mode in MODES_TO_TEST:
        prefix = mode.lower()
        probe = recovery_records[recovery_records["mode"].str.lower().eq(prefix)]
        baseline = trial_summary[
            trial_summary["concurrency_level"].eq(1)
            & trial_summary["mode"].str.lower().eq(prefix)
        ]
        record = probe.iloc[0] if len(probe) == 1 else None
        baseline_ms = baseline["p50_latency_ms"].iloc[0] if len(baseline) == 1 else None
        baseline_ms = None if pd.isna(baseline_ms) else float(baseline_ms)
        result[f"{prefix}_success"] = bool(record["success"]) if record is not None else False
        result[f"{prefix}_latency_ms"] = float(record["latency_ms"]) if record is not None else None
        result[f"{prefix}_anomaly_detected"] = bool(record["anomaly_detected"]) if record is not None else None
        result[f"{prefix}_baseline_p50_ms"] = baseline_ms
        result[f"{prefix}_recovery_latency_limit_ms"] = 2.0 * baseline_ms if baseline_ms is not None else None
        if baseline_ms is None:
            unavailable.append(f"{mode} concurrency-1 successful latency baseline unavailable")
        elif record is None:
            failed.append(f"{mode} recovery response missing")
        elif not bool(record["success"]):
            failed.append(f"{mode} transport failure")
        elif bool(record["anomaly_detected"]):
            failed.append(f"{mode} response anomaly detected")
        elif float(record["latency_ms"]) > 2.0 * baseline_ms:
            failed.append(f"{mode} latency exceeded twice its same-trial baseline")
    if unavailable:
        status, reasons = "not_evaluable", unavailable + failed
    elif failed:
        status, reasons = "failed", failed
    else:
        status, reasons = "passed", ["Both modes passed transport, integrity, and latency checks"]
    result.update({
        "recovery_probe_status": status,
        "recovery_probe_passed": status == "passed",
        "recovery_probe_reason": "; ".join(reasons),
    })
    return result

# Use the shared stage duration for each mode's throughput; summing successful
# mode throughput therefore yields aggregate successful stage throughput.
def build_per_trial_summary(
    scored_requests: pd.DataFrame, stages: pd.DataFrame,
) -> pd.DataFrame:
    """Create one scored row per trial, concurrency, and mode."""
    summary_columns = [
        "trial_id", "trial_number", "concurrency_level", "mode",
        "attempt_count", "success_count", "failure_count", "success_rate",
        "mean_latency_ms", "p50_latency_ms", "p95_latency_ms", "p99_latency_ms",
        "max_latency_ms", "failure_p50_latency_ms", "failure_p95_latency_ms",
        "failure_max_latency_ms", "attempted_throughput_rps",
        "successful_throughput_rps", "failed_completion_rps",
        "anomalous_response_count", "anomalous_response_rate",
    ]
    if scored_requests.empty:
        return pd.DataFrame(columns=summary_columns)
    rows: List[Dict[str, Any]] = []
    for (trial_id, trial_number, stage_id, concurrency_level, mode), group in scored_requests.groupby(
        ["trial_id", "trial_number", "stage_id", "concurrency_level", "mode"], sort=True
    ):
        duration_values = stages.loc[stages["stage_id"].eq(stage_id), "stage_duration_seconds"]
        duration = float(duration_values.iloc[0]) if not duration_values.empty else None
        rows.append({
            "trial_id": trial_id, "trial_number": int(trial_number),
            "concurrency_level": int(concurrency_level), "mode": mode,
            **summarize_request_group(group, duration),
        })
    return pd.DataFrame(rows).sort_values(["trial_number", "concurrency_level", "mode"])


## Live stage and trial orchestration helpers

Define the asynchronous orchestration for scored stages, recovery probes,
and complete trials.

Each scored stage runs the configured paired-mode workload, records its
shared stage duration, checkpoints completed evidence, and then observes the
configured inter-stage cooldown.

After the final scored stage, the trial enters its recovery window. A
lightweight paired-mode recovery probe is sent at the configured offset, and
the remaining recovery-window duration is observed before a subsequent
trial begins.

Recovery requests are diagnostic and are excluded from scored latency,
throughput, transport-success, and anomaly summaries.

In [ ]:
# Execute scored stages, timed recovery probes, and complete trials while
# checkpointing request evidence after every recoverable boundary.
async def run_scored_stage(
    *, trial_id: str, trial_number: int, stage_sequence: int,
    concurrency_level: int, benchmark_run_id: str,
) -> tuple[List[Dict[str, Any]], Dict[str, Any]]:
    """Run one unchanged burst stage and attach protocol identity metadata."""
    stage_id = f"{trial_id}-stage-{stage_sequence:02d}-c{concurrency_level:03d}"
    payload = await run_load_test(
        scenarios, concurrent_users=concurrency_level,
        mode_pair_repeats_per_user=MODE_PAIR_REPEATS_PER_USER,
        timeout=TIMEOUT, modes=MODES_TO_TEST, benchmark_run_id=benchmark_run_id,
        trial_id=trial_id, trial_number=trial_number, stage_id=stage_id,
        stage_sequence=stage_sequence, stage_type="scored",
    )
    for record in payload["results"]:
        record["stage_duration_seconds"] = payload["stage_duration_seconds"]
    metadata = {
        "benchmark_run_id": benchmark_run_id, "trial_id": trial_id,
        "trial_number": trial_number, "stage_id": stage_id,
        "stage_sequence": stage_sequence, "stage_type": "scored",
        "concurrency_level": concurrency_level,
        "stage_cooldown_seconds": STAGE_COOLDOWN_SECONDS,
        **{key: value for key, value in payload.items() if key != "results"},
    }
    return payload["results"], metadata


# Keep recovery probes explicitly typed so they can be excluded from all scored
# latency, throughput, reliability, and anomaly metrics.
async def run_recovery_probe(
    *, trial_id: str, trial_number: int, stage_sequence: int,
    scenario: Dict[str, Any], benchmark_run_id: str,
) -> tuple[List[Dict[str, Any]], Dict[str, Any]]:
    """Send one paired diagnostic on fixed Example 6 with bounded concurrency two."""
    stage_id = f"{trial_id}-recovery-probe"
    payload = await run_load_test(
        [scenario], concurrent_users=1, mode_pair_repeats_per_user=1,
        timeout=TIMEOUT, modes=MODES_TO_TEST, benchmark_run_id=benchmark_run_id,
        trial_id=trial_id, trial_number=trial_number, stage_id=stage_id,
        stage_sequence=stage_sequence, stage_type="recovery_probe",
        configured_max_in_flight=2,
    )
    for record in payload["results"]:
        record["stage_duration_seconds"] = payload["stage_duration_seconds"]
    metadata = {
        "benchmark_run_id": benchmark_run_id, "trial_id": trial_id,
        "trial_number": trial_number, "stage_id": stage_id,
        "stage_sequence": stage_sequence, "stage_type": "recovery_probe",
        "concurrency_level": 1,
        **{key: value for key, value in payload.items() if key != "results"},
    }
    return payload["results"], metadata

async def run_benchmark_trial(
    *, trial_number: int, benchmark_run_id: str,
    sleep_func: Any = asyncio.sleep,
) -> Dict[str, Any]:
    """Execute scored stages, recovery timing, and evidence for one trial."""
    trial_id = build_trial_id(benchmark_run_id, trial_number)
    trial_started_at = datetime.now(timezone.utc).isoformat()
    records: List[Dict[str, Any]] = []
    stages: List[Dict[str, Any]] = []
    recovery_evidence: Dict[str, Any] = {}
    completed_scored_stage_count = 0
    final_scored_stage_completed_at: Optional[str] = None
    print(f"Starting trial {trial_number} of {TRIALS}")
    try:
        # Run scored concurrency stages in fixed order and checkpoint canonical
        # request evidence immediately after each completed stage.
        for stage_sequence, concurrency_level in enumerate(CONCURRENCY_LEVELS, start=1):
            print(f"Starting scored stage {stage_sequence}/{CONFIGURED_STAGE_COUNT} at concurrency {concurrency_level}")
            stage_records, stage = await run_scored_stage(
                trial_id=trial_id, trial_number=trial_number,
                stage_sequence=stage_sequence, concurrency_level=concurrency_level,
                benchmark_run_id=benchmark_run_id,
            )
            records.extend(stage_records)
            stages.append(stage)
            LIVE_REQUEST_RECORDS.extend(stage_records)
            LIVE_STAGE_METADATA.append(stage)
            completed_scored_stage_count += 1
            checkpoint_live_evidence(
                last_completed_stage_id=stage["stage_id"],
                completed_scored_stage_increment=1,
            )
            successes = sum(bool(item["success"]) for item in stage_records)
            print(f"Completed stage: {successes} successful responses, {len(stage_records) - successes} failures")
            if stage_sequence < len(CONCURRENCY_LEVELS):
                next_level = CONCURRENCY_LEVELS[stage_sequence]
                print(f"Cooling down for {STAGE_COOLDOWN_SECONDS} seconds before concurrency {next_level}.")
                await sleep_func(STAGE_COOLDOWN_SECONDS)
        # Measure the recovery window from the final scored stage, issue the fixed
        # diagnostic probe, then honor the remainder of the trial cooldown.
        final_scored_stage_completed_at = stages[-1]["completed_at"]
        print(f"Starting recovery window for trial {trial_number}")
        recovery_window_started = time.perf_counter()
        await sleep_func(RECOVERY_PROBE_AT_SECONDS)
        print(f"Sending recovery probe at {RECOVERY_PROBE_AT_SECONDS} seconds")
        probe_started_at = datetime.now(timezone.utc).isoformat()
        seconds_after_final_stage = time.perf_counter() - recovery_window_started
        probe_records, probe_stage = await run_recovery_probe(
            trial_id=trial_id, trial_number=trial_number,
            stage_sequence=RECOVERY_STAGE_SEQUENCE,
            scenario=next(item for item in scenarios if item["example"] == 6),
            benchmark_run_id=benchmark_run_id,
        )
        probe_completed_at = datetime.now(timezone.utc).isoformat()
        records.extend(probe_records)
        stages.append(probe_stage)
        LIVE_REQUEST_RECORDS.extend(probe_records)
        LIVE_STAGE_METADATA.append(probe_stage)
        checkpoint_live_evidence(last_completed_stage_id=probe_stage["stage_id"])
        trial_scored = pd.DataFrame([r for r in records if r["stage_type"] == "scored"])
        trial_stages = pd.DataFrame([s for s in stages if s["stage_type"] == "scored"])
        trial_summary = build_per_trial_summary(trial_scored, trial_stages)
        recovery_evidence = {
            "trial_id": trial_id, "trial_number": trial_number,
            "probe_started_at": probe_started_at, "probe_completed_at": probe_completed_at,
            "seconds_after_final_stage": seconds_after_final_stage,
            **evaluate_recovery_probe(pd.DataFrame(probe_records), trial_summary),
        }
        print(f"Recovery probe status: {recovery_evidence['recovery_probe_status']}")
        elapsed = time.perf_counter() - recovery_window_started
        remaining_cooldown = max(0.0, TRIAL_COOLDOWN_SECONDS - elapsed)
        print(f"Waiting remaining cooldown time ({remaining_cooldown:.1f} seconds)")
        await sleep_func(remaining_cooldown)
    except asyncio.CancelledError:
        # Preserve the last valid checkpoint and record that execution did not finish.
        LIVE_RUN_MANIFEST["benchmark_status"] = "incomplete"
        LIVE_RUN_MANIFEST["interruption_reason"] = "benchmark execution cancelled"
        LIVE_RUN_MANIFEST["benchmark_interrupted_at"] = datetime.now(timezone.utc).isoformat()
        checkpoint_live_evidence()
        raise
    # Mark unexpected partial runs incomplete while retaining the last canonical
    # checkpoint for diagnosis and later regeneration.
    except Exception as exc:
        trial_error = f"{type(exc).__name__}: {exc}"
        LIVE_RUN_MANIFEST["benchmark_status"] = "incomplete"
        checkpoint_live_evidence()
        print(f"Trial {trial_number} incomplete: {trial_error}")
    else:
        trial_error = None
    trial_completed_at = datetime.now(timezone.utc).isoformat()
    metadata = {
        "trial_id": trial_id, "trial_number": trial_number,
        "trial_started_at": trial_started_at,
        "final_scored_stage_completed_at": final_scored_stage_completed_at,
        "recovery_probe_started_at": recovery_evidence.get("probe_started_at"),
        "recovery_probe_completed_at": recovery_evidence.get("probe_completed_at"),
        "trial_completed_at": trial_completed_at,
        "scored_stage_count": CONFIGURED_STAGE_COUNT,
        "completed_scored_stage_count": completed_scored_stage_count,
        "trial_completed_successfully": completed_scored_stage_count == CONFIGURED_STAGE_COUNT,
        "trial_error": trial_error,
    }
    LIVE_TRIAL_METADATA.append(metadata)
    checkpoint_live_evidence(
        last_completed_trial=trial_number if metadata["trial_completed_successfully"] else None,
    )
    print(f"Trial {trial_number} complete")
    return {"records": records, "stages": stages, "recovery": recovery_evidence, "metadata": metadata}


# Repeat the same protocol in trial order and return execution metadata without
# performing reporting or aggregation in the orchestration layer.
async def run_benchmark_protocol(
    benchmark_started_at: str, sleep_func: Any = asyncio.sleep,
) -> Dict[str, Any]:
    """Run identical trials in fixed order while preserving partial evidence."""
    validate_trial_protocol_configuration()
    outcomes = []
    for trial_number in range(1, TRIALS + 1):
        outcomes.append(await run_benchmark_trial(
            trial_number=trial_number, benchmark_run_id=BENCHMARK_RUN_ID,
            sleep_func=sleep_func,
        ))
    benchmark_completed_at = datetime.now(timezone.utc).isoformat()
    manifest = {
        "benchmark_run_id": BENCHMARK_RUN_ID,
        "target_inhibitor_version": (
            INHIBITOR_API_VERSION
            or "not reported"
        ),
        "target_version_source": (
            "api_connectivity_diagnostic"
        ),
        "benchmark_started_at": benchmark_started_at,
        "benchmark_completed_at": benchmark_completed_at,
        "configured_trials": CONFIGURED_TRIAL_COUNT,
        "completed_trials": sum(item["metadata"]["trial_completed_successfully"] for item in outcomes),
        "concurrency_levels": list(CONCURRENCY_LEVELS),
        "scored_stages_per_trial": CONFIGURED_STAGE_COUNT,
        "stage_cooldown_seconds": STAGE_COOLDOWN_SECONDS,
        "trial_cooldown_seconds": TRIAL_COOLDOWN_SECONDS,
        "recovery_probe_at_seconds": RECOVERY_PROBE_AT_SECONDS,
        "mode_pair_repeats_per_user": MODE_PAIR_REPEATS_PER_USER,
        "modes": MODES_TO_TEST,
        "scenario_input_path": repository_relative_path(SCENARIO_INPUT_PATH,REPOSITORY_ROOT,),
        "results_output_dir": repository_relative_path(run_output_dir,REPOSITORY_ROOT,),
        "scenario_file_hash": hashlib.sha256(SCENARIO_INPUT_PATH.read_bytes()).hexdigest(),
    }
    return {"outcomes": outcomes, "manifest": manifest}


## Formal-run initialization and preflight

Create the unique run directory, calculate the scenario-file hash, initialize
the canonical manifests, and write an empty request-evidence checkpoint
before formal traffic begins.

The run manifest records the deployed Inhibitor version captured by the
connectivity diagnostic, the configured protocol, planned request counts,
artifact schema version, source paths, and checkpoint status.

Review the printed preflight values before starting the live benchmark.

In [ ]:
# Calculate planned request counts and initialize run provenance before the first
# scored request is allowed to start.
scenario_file_hash = hashlib.sha256(SCENARIO_INPUT_PATH.read_bytes()).hexdigest()
planned_requests_per_trial = sum(level * len(MODES_TO_TEST) * MODE_PAIR_REPEATS_PER_USER for level in CONCURRENCY_LEVELS)
total_planned_scored_requests = TRIALS * planned_requests_per_trial
total_planned_recovery_requests = TRIALS * len(MODES_TO_TEST)
run_output_dir = build_run_output_directory(RESULTS_OUTPUT_DIR, BENCHMARK_RUN_ID)
benchmark_started_at = datetime.now(timezone.utc).isoformat()
scenario_manifest = build_scenario_manifest(scenarios, scenario_file_hash)
LIVE_REQUEST_RECORDS: List[Dict[str, Any]] = []
LIVE_STAGE_METADATA: List[Dict[str, Any]] = []
LIVE_TRIAL_METADATA: List[Dict[str, Any]] = []
# Initialize the canonical manifest as running so an interrupted execution cannot be
# mistaken for a completed benchmark.
LIVE_RUN_MANIFEST = {
    "benchmark_run_id": BENCHMARK_RUN_ID, "benchmark_status": "running",
    "target_inhibitor_version": (
        INHIBITOR_API_VERSION
        or "not reported"
    ),
    "target_version_source": (
        "api_connectivity_diagnostic"
    ),
    "benchmark_started_at": benchmark_started_at, "benchmark_completed_at": None,
    "benchmark_type": "progressive_burst_concurrency",
    "configured_trials": TRIALS, "completed_trials": 0,
    "configured_concurrency_levels": list(CONCURRENCY_LEVELS),
    "scored_stages_per_trial": len(CONCURRENCY_LEVELS),
    "completed_scored_stages": 0, "stage_cooldown_seconds": STAGE_COOLDOWN_SECONDS,
    "trial_cooldown_seconds": TRIAL_COOLDOWN_SECONDS,
    "recovery_probe_at_seconds": RECOVERY_PROBE_AT_SECONDS,
    "mode_pair_repeats_per_user": MODE_PAIR_REPEATS_PER_USER,
    "modes": list(MODES_TO_TEST), "scenario_input_path": repository_relative_path(SCENARIO_INPUT_PATH,REPOSITORY_ROOT,),
    "scenario_file_hash": scenario_file_hash, "results_output_dir": repository_relative_path(run_output_dir,REPOSITORY_ROOT,),
    "inhibitor_base_url": INHIBITOR_API_URL, "request_timeout_seconds": TIMEOUT,
    "benchmark_notebook_path": "operations/benchmark_testing/inhibitor_operational_benchmark.ipynb",
    "artifact_schema_version": ARTIFACT_SCHEMA_VERSION,
    "total_planned_scored_requests": total_planned_scored_requests,
    "total_planned_recovery_requests": total_planned_recovery_requests,
    "request_records_checkpointed": 0, "last_checkpoint_at": None,
    "last_completed_trial": None, "last_completed_stage_id": None,
}


# Write request_records.jsonl as canonical request-level evidence; the scored and
# recovery JSONL files produced elsewhere are convenience subsets.
def checkpoint_live_evidence(
    *, last_completed_stage_id: Optional[str] = None,
    completed_scored_stage_increment: int = 0,
    last_completed_trial: Optional[int] = None,
) -> None:
    """Persist canonical evidence before aggregation, charts, or reporting can fail."""
    LIVE_RUN_MANIFEST["completed_scored_stages"] += completed_scored_stage_increment
    LIVE_RUN_MANIFEST["request_records_checkpointed"] = len(LIVE_REQUEST_RECORDS)
    LIVE_RUN_MANIFEST["last_checkpoint_at"] = datetime.now(timezone.utc).isoformat()
    if last_completed_stage_id is not None:
        LIVE_RUN_MANIFEST["last_completed_stage_id"] = last_completed_stage_id
    if last_completed_trial is not None:
        LIVE_RUN_MANIFEST["last_completed_trial"] = last_completed_trial
        LIVE_RUN_MANIFEST["completed_trials"] = last_completed_trial
    write_raw_evidence_artifacts(
        run_dir=run_output_dir, request_records=LIVE_REQUEST_RECORDS,
        run_manifest=LIVE_RUN_MANIFEST,
        stage_metadata=pd.DataFrame(LIVE_STAGE_METADATA),
        trial_metadata=pd.DataFrame(LIVE_TRIAL_METADATA),
    )


# Manifests and an empty canonical checkpoint exist before the first scored request.
write_json(run_output_dir / "scenario_manifest.json", scenario_manifest)
checkpoint_live_evidence()
print("Formal-run preflight")
print(f"- Scenario input path: {SCENARIO_INPUT_PATH.resolve()}")
print(f"- Results output path: {run_output_dir.resolve()}")
print(f"- Inhibitor base URL: {INHIBITOR_API_URL}")
print(f"- Configured trials: {TRIALS}")
print(f"- Configured concurrency levels: {CONCURRENCY_LEVELS}")
print(f"- Total planned scored requests: {total_planned_scored_requests}")
print(f"- Total planned recovery requests: {total_planned_recovery_requests}")
print(f"- Stage cooldown: {STAGE_COOLDOWN_SECONDS} seconds")
print(f"- Trial cooldown: {TRIAL_COOLDOWN_SECONDS} seconds")
print(f"- Recovery probe timing: {RECOVERY_PROBE_AT_SECONDS} seconds")
print(f"- Scenario file hash: {scenario_file_hash}")

## Live benchmark execution

This section sends the formal live API traffic.

The first cell executes every configured scored stage, cooldown, recovery
probe, and trial window. The following cell finalizes the run manifest from
the actual protocol outcome.

Run status is classified as:

- `completed` when all configured trials finish without request failures;
- `completed_with_request_failures` when all trials finish but one or more
  requests fail; or
- `incomplete` when the configured protocol does not fully complete.

Canonical evidence is checkpointed during execution, so an interrupted run
may still retain usable partial request, stage, trial, and manifest evidence.

In [ ]:
# Start the only formal execution cell; running it sends the configured live API
# workload and is intentionally separate from preflight and reporting.
print(
    "Starting the formal live benchmark. "
    "This cell will send API requests."
)
protocol_result = await run_benchmark_protocol(benchmark_started_at)

In [ ]:
# Finalize the live run only after all configured trials have returned.

# Finalize the manifest using the protocol's actual completion timestamp.
protocol_completed_at = protocol_result[
    "manifest"
]["benchmark_completed_at"]

completed_trial_count = sum(
    bool(
        outcome["metadata"][
            "trial_completed_successfully"
        ]
    )
    for outcome in protocol_result["outcomes"]
)

all_trials_completed = (
    completed_trial_count == TRIALS
)

all_request_records = [
    record
    for outcome in protocol_result["outcomes"]
    for record in outcome["records"]
]

request_failure_count = sum(
    not bool(record["success"])
    for record in all_request_records
)

if all_trials_completed:
    # Distinguish a completed protocol with request failures from a fully successful run.
    LIVE_RUN_MANIFEST["benchmark_status"] = (
        "completed_with_request_failures"
        if request_failure_count
        else "completed"
    )
else:
    # Preserve partial evidence without presenting the protocol as complete.
    LIVE_RUN_MANIFEST["benchmark_status"] = "incomplete"

# Preserve the timestamp recorded when the protocol actually finished.
LIVE_RUN_MANIFEST["benchmark_completed_at"] = (
    protocol_completed_at
)

LIVE_RUN_MANIFEST["completed_trials"] = (
    completed_trial_count
)

LIVE_RUN_MANIFEST["request_failure_count"] = (
    request_failure_count
)

# Persist the final status and the last canonical evidence checkpoint.
checkpoint_live_evidence()

print(
    "Final benchmark status: "
    f"{LIVE_RUN_MANIFEST['benchmark_status']}"
)

print(
    "Completed trials: "
    f"{completed_trial_count}/{TRIALS}"
)

print(
    "Request failures: "
    f"{request_failure_count}"
)

## Normalize live results for reporting

Convert the completed live protocol output into the canonical in-memory
tables used by the reporting pipeline.

At this point, request records, stage metadata, trial metadata, recovery
evidence, and the run manifest represent the live execution path. The next
section can replace these inputs with evidence loaded from a saved run.

In [ ]:
# Convert the completed live protocol into canonical in-memory reporting inputs.
all_results = [record for outcome in protocol_result["outcomes"] for record in outcome["records"]]
stage_metadata = [stage for outcome in protocol_result["outcomes"] for stage in outcome["stages"]]
trial_metadata = [outcome["metadata"] for outcome in protocol_result["outcomes"]]
recovery_summaries = [outcome["recovery"] for outcome in protocol_result["outcomes"] if outcome["recovery"]]
requests_df = pd.DataFrame(all_results, columns=REQUEST_RECORD_FIELDS)
stage_metadata_df = pd.DataFrame(stage_metadata)
trial_metadata_df = pd.DataFrame(trial_metadata)
run_manifest = LIVE_RUN_MANIFEST

## Regenerating reports from a saved run

To revise summaries, charts, findings, or report wording without sending new
API requests, set `EXISTING_RUN_DIR` to an existing run directory and execute
from the next cell downward.

Build the path from `REPOSITORY_ROOT` so execution does not depend on the
notebook's current working directory.

The canonical regeneration inputs are:

- `request_records.jsonl`;
- `run_manifest.json`; and
- `scenario_manifest.json`.

All CSV summaries, charts, deterministic findings, convenience JSONL
subsets, and the Markdown report are derived from those canonical inputs.

Leave `EXISTING_RUN_DIR` as `None` to continue from live results already held
in memory. Do not rerun the live benchmark execution cells for report-only
work.


In [ ]:
# Leave as None to use the live benchmark results already created in this session.
# Select saved-run regeneration when configured; loading canonical evidence and
# rebuilding reports sends no API requests.
EXISTING_RUN_DIR: Path | None =  None
# EXISTING_RUN_DIR = (
#     REPOSITORY_ROOT
#     / "operations"
#     / "benchmark_testing"
#     / "operational_results"
#     / "<benchmark_run_id>"
# )

if EXISTING_RUN_DIR is not None:
    # Loading canonical JSONL replaces reporting inputs and sends no API requests.
    loaded_run = load_existing_benchmark_run(EXISTING_RUN_DIR)
    requests_df = loaded_run["request_records_df"]
    run_manifest = loaded_run["run_manifest"]
    scenario_manifest = loaded_run["scenario_manifest"]
    run_output_dir = loaded_run["run_dir"]
    reporting_source = "loaded_raw_evidence"
    print(f"Loaded existing benchmark run for reporting: {run_output_dir.resolve()}")
else:
    required_live_variables = ["requests_df", "run_manifest", "scenario_manifest", "run_output_dir"]
    missing_live_variables = [name for name in required_live_variables if name not in globals()]
    if missing_live_variables:
        raise RuntimeError(
            "No existing run was configured and live benchmark results are unavailable. "
            f"Missing variables: {missing_live_variables}"
        )
    reporting_source = "live_memory"
    print("Using live benchmark results already present in memory.")

# Both paths rebuild every derived reporting input from canonical request records.
# Rebuild reporting inputs from canonical request_records.jsonl and keep recovery
# probes separate from the scored population.
requests_df = normalize_request_records_dataframe(requests_df)
scored_requests_df = requests_df[requests_df["stage_type"].eq("scored")].copy()
recovery_requests_df = requests_df[requests_df["stage_type"].eq("recovery_probe")].copy()
progressive_results_df = scored_requests_df
progressive_summary_df = rebuild_per_trial_summary(scored_requests_df)
cross_trial_summary_df = rebuild_cross_trial_summary(progressive_summary_df)
MODES_TO_TEST = list(run_manifest["modes"])
CONCURRENCY_LEVELS = list(run_manifest["configured_concurrency_levels"])
recovery_summary_artifact_df = rebuild_recovery_summary(
    recovery_requests_df, progressive_summary_df, MODES_TO_TEST,
)


## Summary calculations and display

Rebuild and display the per-trial, cross-trial, and recovery summaries from
the active canonical request evidence.

The active source may be either:

- the live run held in memory; or
- a saved run loaded from canonical artifacts.

For a one-trial run, cross-trial values are descriptive and match the single
contributing trial. With multiple trials, cross-trial summaries report
statistics across the per-trial rows rather than blindly pooling all raw
request latencies.

In [ ]:
# Display the summaries already rebuilt from either live or loaded raw evidence.
print(
    "Per-trial scored summary "
    "(trial × concurrency × mode):"
)

print(
    progressive_summary_df.to_markdown(
        index=False
    )
)

# State the repeatability boundary explicitly when only one trial contributes.
configured_trial_count = int(
    run_manifest["configured_trials"]
)

if configured_trial_count == 1:
    cross_trial_note = (
        "This run contains one contributing trial. "
        "Cross-trial median and range values are therefore identical. "
        "They become comparative when multiple trials are configured."
    )
else:
    cross_trial_note = (
        f"This run contains {configured_trial_count} contributing trials. "
        "Cross-trial summaries report medians and ranges."
    )

print(cross_trial_note)

print(
    "Cross-trial summary "
    "(median and range across per-trial rows):"
)

print(
    cross_trial_summary_df.to_markdown(
        index=False
    )
)

print(
    "Recovery summary "
    "(one diagnostic row per trial):"
)

recovery_display_columns = [
    "trial_number",
    "performance_success",
    "insight_success",
    "performance_latency_ms",
    "insight_latency_ms",
    "performance_baseline_p50_ms",
    "insight_baseline_p50_ms",
    "recovery_probe_status",
    "recovery_probe_reason",
]

print(
    recovery_summary_artifact_df.reindex(
        columns=recovery_display_columns
    ).to_markdown(
        index=False
    )
)

## Evidence assembly and deterministic summaries

Build the formal latency, anomaly, recovery, and reporting tables from
canonical request records.

Scored requests and recovery probes remain separate:

- scored requests contribute to formal latency, throughput, transport
  reliability, and anomaly summaries;
- recovery requests contribute only to the post-burst diagnostic assessment.

Successful-response latency and failed-request latency are not combined.
Configured anomalies are counted only among transport-successful responses.

All summaries are derived without modifying the original request records or
the protocol's transport, anomaly, orchestration, and recovery rules.


In [ ]:
# Build deterministic latency and anomaly tables from scored requests only; anomaly
# checks describe response shape and configured minima, not semantic correctness.
latency_summary_df = progressive_summary_df.reindex(columns=[
    "trial_number", "concurrency_level", "mode", "success_count", "mean_latency_ms",
    "p50_latency_ms", "p95_latency_ms", "p99_latency_ms", "max_latency_ms",
    "failure_p50_latency_ms", "failure_p95_latency_ms", "failure_max_latency_ms",
])
anomaly_summary_artifact_df, anomaly_breakdown_df = build_anomaly_tables(scored_requests_df)

print("Successful-response latency and separate failure latency:")
print(latency_summary_df.to_markdown(index=False))
print("Concise anomaly summary:")
print(anomaly_summary_artifact_df.to_markdown(index=False))
# Suppress an empty anomaly breakdown while preserving the concise summary.
if anomaly_breakdown_df.empty:
    print("No response anomalies were detected among transport-successful responses.")
else:
    print(anomaly_breakdown_df.to_markdown(index=False))


## Formal charts

Generate deterministic charts from the active reporting tables.

The chart set includes:

- successful-response latency by concurrency;
- successful throughput by concurrency and mode;
- transport success rate by concurrency;
- configured response-anomaly rate by concurrency;
- recovery latency relative to the same-trial concurrency-1 baseline; and
- successful-latency histograms for groups with enough observations.

Scored charts exclude recovery requests. Failed requests are excluded from
successful-latency distributions and are reported separately.

A failure-specific chart is created only when scored failures exist.
Histograms are omitted for groups below the configured minimum observation
count. Missing latency is never plotted as zero.


In [ ]:
# Generate charts deterministically from rebuilt summaries; the reporting helper
# conditionally omits failure charts and histograms when supporting data is absent.
plot_paths = generate_formal_charts(
    cross_trial_summary_df, scored_requests_df, recovery_summary_artifact_df,
    list(CONCURRENCY_LEVELS), run_output_dir / "plots",
)
print(f"Generated {len(plot_paths)} formal chart files.")


## Artifact export and client report

Build the deterministic findings, eleven-section client-facing Markdown
report, report manifest, convenience evidence subsets, flattened CSV views,
and final reporting artifacts.

Artifact roles are separated deliberately:

- `request_records.jsonl` is the canonical request-level evidence source;
- `scored_request_records.jsonl` and
  `recovery_request_records.jsonl` are convenience subsets;
- CSV files provide flattened analysis views;
- manifests preserve workload, protocol, version, source, and reporting
  provenance;
- summaries and plots are deterministic derived artifacts; and
- `benchmark_report.md` is the client-facing interpretation of the run.

When regenerating from a saved run, canonical request evidence and execution
provenance are preserved. The report manifest records the report revision,
generation timestamp, reporting source, and reporting-code version.


In [ ]:
# Preserve the benchmark execution manifest and timestamps during report regeneration.
BENCHMARK_RUN_ID = run_manifest["benchmark_run_id"]
TRIALS = int(run_manifest["configured_trials"])
completed_trials = int(run_manifest.get("completed_trials", 0))
transport_failures = int((~scored_requests_df.get("success", pd.Series(dtype=bool)).fillna(False)).sum())
if reporting_source == "live_memory":
    run_manifest["benchmark_completed_at"] = protocol_result["manifest"]["benchmark_completed_at"]
    run_manifest["benchmark_duration_seconds"] = (
        datetime.fromisoformat(run_manifest["benchmark_completed_at"])
        - datetime.fromisoformat(run_manifest["benchmark_started_at"])
    ).total_seconds()
    if completed_trials < TRIALS:
        run_manifest["benchmark_status"] = "incomplete"
    elif transport_failures:
        run_manifest["benchmark_status"] = "completed_with_request_failures"
    else:
        run_manifest["benchmark_status"] = "completed"
    run_manifest["request_records_checkpointed"] = len(requests_df)
    run_manifest["last_checkpoint_at"] = datetime.now(timezone.utc).isoformat()
    write_raw_evidence_artifacts(
        run_dir=run_output_dir, request_records=requests_df.to_dict("records"),
        run_manifest=run_manifest, stage_metadata=stage_metadata_df,
        trial_metadata=trial_metadata_df,
    )

# Derive findings and report text deterministically from canonical evidence rather
# than introducing manual interpretation during regeneration.
findings = build_deterministic_findings(
    progressive_summary_df, cross_trial_summary_df, scored_requests_df,
    recovery_summary_artifact_df,
)
artifact_names = [
    "run_manifest.json", "scenario_manifest.json", "request_records.jsonl",
    "request_records.csv", "scored_request_records.jsonl", "recovery_request_records.jsonl",
    "stage_metadata.csv", "trial_metadata.csv", "per_trial_summary.csv",
    "cross_trial_summary.csv", "recovery_summary.csv", "anomaly_summary.csv",
    "anomaly_breakdown.csv", "benchmark_report.md", "report_manifest.json", "plots/",
]
report_content = render_benchmark_report(
    manifest=run_manifest, scenario_manifest=scenario_manifest,
    per_trial=progressive_summary_df, cross_trial=cross_trial_summary_df,
    scored=scored_requests_df, recovery=recovery_summary_artifact_df,
    anomaly_summary=anomaly_summary_artifact_df, anomaly_breakdown=anomaly_breakdown_df,
    findings=findings, plot_paths=plot_paths, artifact_names=artifact_names,
)
# Increment report revision metadata without altering the preserved execution
# manifest or original benchmark timestamps.
previous_report_manifest_path = run_output_dir / "report_manifest.json"
report_revision = 1
if previous_report_manifest_path.exists():
    previous_report_manifest = json.loads(previous_report_manifest_path.read_text(encoding="utf-8"))
    report_revision = int(previous_report_manifest.get("report_revision", 0)) + 1
report_manifest = {
    "source_benchmark_run_id": BENCHMARK_RUN_ID,
    "source_run_directory": run_manifest["results_output_dir"],
    "report_generated_at": datetime.now(timezone.utc).isoformat(),
    "report_revision": report_revision,
    "reporting_code_version": ARTIFACT_SCHEMA_VERSION,
    "reporting_source": reporting_source,
}
# Export derived tables, convenience request subsets, report, manifest, and plots
# only after all reporting inputs have been rebuilt.
artifact_paths = write_derived_reporting_artifacts(
    run_dir=run_output_dir, requests=requests_df, scored=scored_requests_df,
    recovery_requests=recovery_requests_df, per_trial=progressive_summary_df,
    cross_trial=cross_trial_summary_df, recovery_summary=recovery_summary_artifact_df,
    anomaly_summary=anomaly_summary_artifact_df, anomaly_breakdown=anomaly_breakdown_df,
    report=report_content, report_manifest=report_manifest,
    write_raw_splits=reporting_source == "live_memory",
)
print("Deterministic findings:")
print("\n".join(f"- {finding}" for finding in findings))


## Completion summary

Print a concise handoff containing the benchmark run identifier, final run
status, completed trial count, scored and recovery request counts, transport
failure count, configured anomaly count, recovery status, report path, and
artifact directory.

This output does not print credentials or raw response bodies.


In [ ]:
# Summarize completion and exported artifact locations without changing evidence
# or making additional service requests.
recovery_status = ", ".join(sorted(set(recovery_summary_artifact_df.get("recovery_probe_status", pd.Series(dtype=str)).dropna()))) or "not evaluable"
responses_with_anomalies = int(anomaly_summary_artifact_df.get("anomalous_responses", pd.Series(dtype=int)).sum())
print(f"Benchmark run ID: {BENCHMARK_RUN_ID}")
# Distinct quote styles keep the completion summary valid Python.
print(f"Run status: {run_manifest['benchmark_status']}")
print(f"Completed trials: {completed_trials}/{TRIALS}")
print(f"Scored request records written: {len(scored_requests_df)}")
print(f"Recovery request records written: {len(recovery_requests_df)}")
print(f"Transport failures: {transport_failures}")
print(f"Responses with anomalies: {responses_with_anomalies}")
print(f"Recovery status: {recovery_status}")
print(f"Report path: {artifact_paths['benchmark_report.md']}")
print(f"Artifact directory: {run_output_dir}")
